In [ ]:
import os
import csv
import random
import sqlite3
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

In [ ]:
os.makedirs("data", exist_ok=True)

CATEGORIES = {
    "Electronics": ["Smartphone", "Laptop", "Headphones", "Smartwatch", "Tablet", "Camera", "Charger"],
    "Clothing": ["T-Shirt", "Jeans", "Jacket", "Sneakers", "Dress", "Socks", "Sweater"],
    "Home": ["Lamp", "Cushion", "Blender", "Coffee Maker", "Vase", "Rug", "Towels"],
    "Books": ["Fiction Novel", "Sci-Fi Book", "Biography", "History Book", "Cookbook", "Comic Book"]
}
REGIONS = ["US-E", "US-W", "EU-W", "APAC-S", "APAC-N"]
STATUSES = ["PLACED", "SHIPPED", "DELIVERED", "CANCELLED", "RETURNED"]
CUSTOMER_TYPES = ["REGULAR", "PREMIUM", "VIP"]
NAMES = ["Aarav", "Aditi", "Amit", "Ananya", "Arjun", "Deepak", "Divya", "Ganesh", "Isha", "Karan", 
         "Kavita", "Manish", "Meera", "Nikhil", "Pooja", "Pranav", "Priya", "Rahul", "Riya", "Sanjay",
         "Shreya", "Siddharth", "Sneha", "Sunita", "Vikram", "Neha", "Rohan", "Suresh", "Anjali", "Kiran"]
SURNAMES = ["Sharma", "Verma", "Gupta", "Patel", "Mehta", "Reddy", "Nair", "Joshi", "Singh", "Rao",
            "Gupta", "Das", "Choudhury", "Bose", "Pillai", "Iyer", "Sen", "Roy", "Mishra", "Desai"]

customers = []
customer_ids = [f"CUST{str(i).zfill(4)}" for i in range(1, 550)]
for cust_id in customer_ids:
    first_name = random.choice(NAMES)
    last_name = random.choice(SURNAMES)
    name = f"{first_name} {last_name}"
    is_invalid_email = random.random() < 0.02
    if is_invalid_email:
        email = f"{first_name.lower()}.{last_name.lower()}gmail.com" if random.random() < 0.5 else f"{first_name.lower()}.{last_name.lower()}@"
    else:
        email = f"{first_name.lower()}.{last_name.lower()}@example.com"
    reg_date = (datetime.now() - timedelta(days=random.randint(100, 700))).strftime("%Y-%m-%d")
    customers.append([cust_id, name, email, reg_date, random.choice(CUSTOMER_TYPES)])

pd.DataFrame(customers, columns=["customer_id", "customer_name", "email", "registration_date", "customer_type"]).to_csv("data/customers.csv", index=False)

products = []
product_ids = [f"PROD{str(i).zfill(4)}" for i in range(1, 520)]
for prod_id in product_ids:
    category = random.choice(list(CATEGORIES.keys()))
    subcategory = random.choice(CATEGORIES[category])
    raw_name = f"{subcategory} {random.randint(100, 999)}"
    casing_issue = random.random()
    if casing_issue < 0.10:
        product_name = f"  {raw_name.upper()}  "
    elif casing_issue < 0.20:
        product_name = "".join(c.upper() if i % 2 == 0 else c.lower() for i, c in enumerate(raw_name))
    else:
        product_name = raw_name
    cost_price = round(random.uniform(5.0, 500.0), 2)
    products.append([prod_id, product_name, category, subcategory, cost_price])

pd.DataFrame(products, columns=["product_id", "product_name", "category", "subcategory", "cost_price"]).to_csv("data/products.csv", index=False)

orders = []
order_ids = [f"ORD{str(i).zfill(5)}" for i in range(1, 650)]
for ord_id in order_ids:
    cust_id = "" if random.random() < 0.05 else random.choice(customer_ids)
    status = random.choice(STATUSES)
    region = random.choice(REGIONS)
    order_dt = datetime.now() - timedelta(days=random.randint(1, 500), hours=random.randint(0, 23), minutes=random.randint(0, 59))
    date_str = order_dt.strftime("%d-%m-%Y") if random.random() < 0.08 else order_dt.strftime("%Y-%m-%d %H:%M:%S")
    orders.append([ord_id, cust_id, date_str, status, region])

pd.DataFrame(orders, columns=["order_id", "customer_id", "order_date", "status", "region_code"]).to_csv("data/orders.csv", index=False)

order_items = []
item_id = 1
for ord_id in order_ids:
    for _ in range(random.randint(1, 3)):
        prod = random.choice(products)
        unit_price = round(prod[4] * random.uniform(1.15, 1.40), 2)
        quantity = -random.randint(1, 3) if random.random() < 0.03 else random.randint(1, 5)
        discount = round(random.uniform(0, 20), 1) if random.random() < 0.4 else 0.0
        order_items.append([f"ITEM{str(item_id).zfill(5)}", ord_id, prod[0], quantity, unit_price, discount])
        item_id += 1

for _ in range(15):
    prod = random.choice(products)
    order_items.append([f"ITEM{str(item_id).zfill(5)}", f"ORD{random.randint(99999, 999999)}", prod[0], random.randint(1, 5), round(prod[4] * 1.25, 2), 0.0])
    item_id += 1

pd.DataFrame(order_items, columns=["item_id", "order_id", "product_id", "quantity", "unit_price", "discount_percent"]).to_csv("data/order_items.csv", index=False)

In [ ]:
orders_df = pd.read_csv("data/orders.csv")
order_items_df = pd.read_csv("data/order_items.csv")
products_df = pd.read_csv("data/products.csv")
customers_df = pd.read_csv("data/customers.csv")

def clean_orders():
    global orders_df
    orders_df["customer_id"] = orders_df["customer_id"].fillna("UNKNOWN")
    cleaned_dates = []
    for idx, row in orders_df.iterrows():
        date_str = str(row["order_date"]).strip()
        if len(date_str) == 10 and date_str[2] == '-' and date_str[5] == '-':
            parsed_date = pd.to_datetime(date_str, format="%d-%m-%Y")
        else:
            parsed_date = pd.to_datetime(date_str)
        cleaned_dates.append(parsed_date.strftime("%Y-%m-%d %H:%M:%S"))
    orders_df["order_date"] = cleaned_dates

def clean_products():
    global products_df
    products_df["product_name"] = products_df["product_name"].str.strip().str.title()

def validate_emails():
    global customers_df
    invalid_customers = []
    for idx, row in customers_df.iterrows():
        email = str(row["email"])
        if "@" not in email or email.endswith("@") or email.split("@")[1] == "":
            invalid_customers.append(row["customer_id"])
    return invalid_customers

def check_referential_integrity():
    global order_items_df, orders_df
    valid_order_ids = set(orders_df["order_id"])
    orphan_items = order_items_df[~order_items_df["order_id"].isin(valid_order_ids)]
    cleaned_order_items = order_items_df[order_items_df["order_id"].isin(valid_order_ids)]
    cleaned_order_items.to_csv("data/cleaned/order_items.csv", index=False)
    return orphan_items

os.makedirs("data/cleaned", exist_ok=True)
clean_orders()
clean_products()
invalid_custs = validate_emails()
orphans = check_referential_integrity()

orders_df.to_csv("data/cleaned/orders.csv", index=False)
products_df.to_csv("data/cleaned/products.csv", index=False)
customers_df.to_csv("data/cleaned/customers.csv", index=False)

In [ ]:
conn = sqlite3.connect("data/ecommerce.db")
cursor = conn.cursor()

cursor.execute("DROP TABLE IF EXISTS order_items;")
cursor.execute("DROP TABLE IF EXISTS orders;")
cursor.execute("DROP TABLE IF EXISTS products;")
cursor.execute("DROP TABLE IF EXISTS customers;")

cursor.execute("""
CREATE TABLE customers (
    customer_id TEXT PRIMARY KEY,
    customer_name TEXT,
    email TEXT,
    registration_date TEXT,
    customer_type TEXT
);""")

cursor.execute("""
CREATE TABLE products (
    product_id TEXT PRIMARY KEY,
    product_name TEXT,
    category TEXT,
    subcategory TEXT,
    cost_price REAL
);""")

cursor.execute("""
CREATE TABLE orders (
    order_id TEXT PRIMARY KEY,
    customer_id TEXT,
    order_date TEXT,
    status TEXT,
    region_code TEXT
);""")

cursor.execute("""
CREATE TABLE order_items (
    item_id TEXT PRIMARY KEY,
    order_id TEXT,
    product_id TEXT,
    quantity INTEGER,
    unit_price REAL,
    discount_percent REAL,
    FOREIGN KEY(order_id) REFERENCES orders(order_id),
    FOREIGN KEY(product_id) REFERENCES products(product_id)
);""")

pd.read_csv("data/cleaned/customers.csv").to_sql("customers", conn, if_exists="append", index=False)
pd.read_csv("data/cleaned/products.csv").to_sql("products", conn, if_exists="append", index=False)
pd.read_csv("data/cleaned/orders.csv").to_sql("orders", conn, if_exists="append", index=False)
pd.read_csv("data/cleaned/order_items.csv").to_sql("order_items", conn, if_exists="append", index=False)

conn.commit()

In [ ]:
def run_query(sql):
    df = pd.read_sql_query(sql, conn)
    display(df.head(10))

run_query("""
    SELECT p.category, ROUND(SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent/100.0)), 2) AS total_revenue
    FROM order_items oi
    JOIN products p ON oi.product_id = p.product_id
    GROUP BY p.category
    ORDER BY total_revenue DESC;
""")

run_query("""
    SELECT c.customer_id, c.customer_name, ROUND(SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent/100.0)), 2) AS total_order_value
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    JOIN customers c ON o.customer_id = c.customer_id
    GROUP BY c.customer_id, c.customer_name
    ORDER BY total_order_value DESC LIMIT 10;
""")

run_query("""
    SELECT STRFTIME('%Y-%m', order_date) AS order_month, COUNT(DISTINCT order_id) AS total_orders
    FROM orders
    WHERE order_date >= DATE('now', '-12 month')
    GROUP BY order_month ORDER BY order_month DESC;
""")

run_query("""
    SELECT DISTINCT c.customer_id, c.customer_name
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    WHERE o.customer_id != 'UNKNOWN' AND o.customer_id NOT IN (SELECT DISTINCT customer_id FROM orders WHERE status = 'DELIVERED');
""")

run_query("""
    SELECT p.product_id, p.product_name, SUM(oi.quantity) AS net_quantity
    FROM order_items oi
    JOIN products p ON oi.product_id = p.product_id
    GROUP BY p.product_id, p.product_name HAVING net_quantity < 0;
""")

run_query("""
    SELECT p.category, SUM(CASE WHEN oi.quantity < 0 THEN ABS(oi.quantity) ELSE 0 END) AS returned_items, SUM(ABS(oi.quantity)) AS total_items,
           ROUND(SUM(CASE WHEN oi.quantity < 0 THEN ABS(oi.quantity) ELSE 0 END) * 100.0 / SUM(ABS(oi.quantity)), 2) AS return_rate_pct
    FROM order_items oi
    JOIN products p ON oi.product_id = p.product_id
    GROUP BY p.category ORDER BY return_rate_pct DESC;
""")

run_query("""
    WITH daily_rev AS (
        SELECT o.region_code, DATE(o.order_date) AS order_day, SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent/100.0)) AS daily_revenue
        FROM orders o JOIN order_items oi ON o.order_id = oi.order_id GROUP BY o.region_code, order_day
    )
    SELECT region_code, order_day, ROUND(daily_revenue, 2) AS daily_revenue,
           ROUND(SUM(daily_revenue) OVER (PARTITION BY region_code ORDER BY order_day), 2) AS running_total
    FROM daily_rev ORDER BY region_code, order_day;
""")

run_query("""
    WITH prod_rev AS (
        SELECT p.category, p.product_name, SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent/100.0)) AS total_revenue
        FROM products p JOIN order_items oi ON p.product_id = oi.product_id GROUP BY p.category, p.product_name
    )
    SELECT category, product_name, ROUND(total_revenue, 2) AS total_revenue,
           DENSE_RANK() OVER (PARTITION BY category ORDER BY total_revenue DESC) AS rank_in_category
    FROM prod_rev;
""")

run_query("""
    WITH ord_lag AS (
        SELECT customer_id, order_date, LAG(order_date) OVER (PARTITION BY customer_id ORDER BY order_date) AS previous_order_date
        FROM orders WHERE customer_id != 'UNKNOWN'
    ), gap_calc AS (
        SELECT customer_id, order_date, previous_order_date, ROUND(JULIANDAY(order_date) - JULIANDAY(previous_order_date), 1) AS days_gap
        FROM ord_lag WHERE previous_order_date IS NOT NULL
    ), avg_gaps AS (
        SELECT customer_id, AVG(days_gap) AS avg_gap FROM gap_calc GROUP BY customer_id
    )
    SELECT g.customer_id, g.order_date, g.previous_order_date, g.days_gap, CASE WHEN a.avg_gap > 30 THEN 'At Risk' ELSE 'Active' END AS status_flag
    FROM gap_calc g JOIN avg_gaps a ON g.customer_id = a.customer_id;
""")

run_query("""
    WITH cust_monthly_rev AS (
        SELECT o.customer_id, STRFTIME('%Y-%m', o.order_date) AS month_str, SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent/100.0)) AS monthly_revenue
        FROM orders o JOIN order_items oi ON o.order_id = oi.order_id WHERE o.customer_id != 'UNKNOWN' GROUP BY o.customer_id, month_str
    ), cust_category AS (
        SELECT customer_id, month_str, CASE WHEN monthly_revenue > 10000 THEN 'High' WHEN monthly_revenue >= 5000 THEN 'Medium' ELSE 'Low' END AS customer_segment
        FROM cust_monthly_rev
    )
    SELECT month_str, customer_segment, COUNT(DISTINCT customer_id) AS customer_count
    FROM cust_category GROUP BY month_str, customer_segment ORDER BY month_str DESC, customer_segment;
""")

run_query("""
    WITH cust_ltv AS (
        SELECT o.customer_id, SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent/100.0)) AS total_value
        FROM orders o JOIN order_items oi ON o.order_id = oi.order_id WHERE o.customer_id != 'UNKNOWN' GROUP BY o.customer_id
    ), quartiles AS (
        SELECT customer_id, total_value, NTILE(4) OVER (ORDER BY total_value DESC) AS quartile FROM cust_ltv
    )
    SELECT customer_id, ROUND(total_value, 2) AS total_value, quartile,
           CASE WHEN quartile = 1 THEN 'Platinum' WHEN quartile = 2 THEN 'Gold' WHEN quartile = 3 THEN 'Silver' ELSE 'Bronze' END AS quartile_label
    FROM quartiles;
""")

run_query("""
    WITH monthly_revenue AS (
        SELECT CAST(STRFTIME('%Y', o.order_date) AS INTEGER) AS rev_year, CAST(STRFTIME('%m', o.order_date) AS INTEGER) AS rev_month,
               SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent/100.0)) AS revenue
        FROM orders o JOIN order_items oi ON o.order_id = oi.order_id GROUP BY rev_year, rev_month
    )
    SELECT curr.rev_year AS year, curr.rev_month AS month, ROUND(curr.revenue, 2) AS revenue, ROUND(prev.revenue, 2) AS prev_year_revenue,
           CASE WHEN prev.revenue IS NULL OR prev.revenue = 0 THEN NULL ELSE ROUND(((curr.revenue - prev.revenue) * 100.0 / prev.revenue), 2) END AS yoy_growth_percent
    FROM monthly_revenue curr
    LEFT JOIN monthly_revenue prev ON curr.rev_year = prev.rev_year + 1 AND curr.rev_month = prev.rev_month;
""")

run_query("""
    WITH customer_purchases AS (
        SELECT o.customer_id, p.category, o.order_date,
               ROW_NUMBER() OVER (PARTITION BY o.customer_id ORDER BY o.order_date ASC, oi.item_id ASC) AS rn_first,
               ROW_NUMBER() OVER (PARTITION BY o.customer_id ORDER BY o.order_date DESC, oi.item_id DESC) AS rn_last
        FROM orders o JOIN order_items oi ON o.order_id = oi.order_id JOIN products p ON oi.product_id = p.product_id WHERE o.customer_id != 'UNKNOWN'
    ), first_purchase AS (
        SELECT customer_id, category AS first_category FROM customer_purchases WHERE rn_first = 1
    ), last_purchase AS (
        SELECT customer_id, category AS last_category FROM customer_purchases WHERE rn_last = 1
    )
    SELECT f.customer_id, f.first_category, l.last_category, CASE WHEN f.first_category = l.last_category THEN 'No' ELSE 'Yes' END AS category_shift
    FROM first_purchase f JOIN last_purchase l ON f.customer_id = l.customer_id;
""")

run_query("""
    WITH customer_revenue AS (
        SELECT o.customer_id, SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent/100.0)) AS revenue
        FROM orders o JOIN order_items oi ON o.order_id = oi.order_id WHERE o.customer_id != 'UNKNOWN' GROUP BY o.customer_id
    ), total_rev AS (
        SELECT SUM(revenue) AS total_revenue FROM customer_revenue
    ), cum_rev AS (
        SELECT customer_id, revenue, SUM(revenue) OVER (ORDER BY revenue DESC) AS cumulative_revenue FROM customer_revenue
    )
    SELECT c.customer_id, ROUND(c.revenue, 2) AS revenue, ROUND(c.cumulative_revenue, 2) AS cumulative_revenue,
           ROUND((c.cumulative_revenue * 100.0 / t.total_revenue), 2) AS cumulative_percent
    FROM cum_rev c CROSS JOIN total_rev t;
""")

run_query("""
    WITH customer_cohort AS (
        SELECT customer_id, STRFTIME('%Y-%m', registration_date) AS cohort_month FROM customers
    ), customer_orders AS (
        SELECT o.customer_id, STRFTIME('%Y-%m', o.order_date) AS order_month, cc.cohort_month,
               (CAST(STRFTIME('%Y', o.order_date) AS INTEGER) - CAST(STRFTIME('%Y', cc.cohort_month || '-01') AS INTEGER)) * 12 +
               (CAST(STRFTIME('%m', o.order_date) AS INTEGER) - CAST(STRFTIME('%m', cc.cohort_month || '-01') AS INTEGER)) AS month_diff
        FROM orders o JOIN customer_cohort cc ON o.customer_id = cc.customer_id WHERE o.status != 'CANCELLED'
    ), cohort_sizes AS (
        SELECT cohort_month, COUNT(DISTINCT customer_id) AS cohort_size FROM customer_cohort GROUP BY cohort_month
    ), retention AS (
        SELECT cohort_month,
               COUNT(DISTINCT CASE WHEN month_diff = 0 THEN customer_id END) AS month_0,
               COUNT(DISTINCT CASE WHEN month_diff = 1 THEN customer_id END) AS month_1,
               COUNT(DISTINCT CASE WHEN month_diff = 2 THEN customer_id END) AS month_2,
               COUNT(DISTINCT CASE WHEN month_diff = 3 THEN customer_id END) AS month_3
        FROM customer_orders GROUP BY cohort_month
    )
    SELECT r.cohort_month, s.cohort_size, r.month_0,
           r.month_1, ROUND((r.month_1 * 100.0 / s.cohort_size), 2) AS month_1_retention_pct,
           r.month_2, ROUND((r.month_2 * 100.0 / s.cohort_size), 2) AS month_2_retention_pct,
           r.month_3, ROUND((r.month_3 * 100.0 / s.cohort_size), 2) AS month_3_retention_pct
    FROM retention r JOIN cohort_sizes s ON r.cohort_month = s.cohort_month;
""")

run_query("""
    SELECT oi1.product_id AS product_a, p1.product_name AS product_a_name, 
           oi2.product_id AS product_b, p2.product_name AS product_b_name, COUNT(*) AS times_bought_together
    FROM order_items oi1
    JOIN order_items oi2 ON oi1.order_id = oi2.order_id AND oi1.product_id < oi2.product_id
    JOIN products p1 ON oi1.product_id = p1.product_id
    JOIN products p2 ON oi2.product_id = p2.product_id
    GROUP BY oi1.product_id, oi2.product_id ORDER BY times_bought_together DESC;
""")

In [ ]:
def test_order_id_not_in_orders():
    try:
        cursor.execute("INSERT INTO order_items (item_id, order_id, product_id, quantity, unit_price, discount_percent) VALUES ('ITEM_ERR', 'ORD_ERR', 'PROD0001', 1, 10.0, 0.0)")
        conn.commit()
    except sqlite3.IntegrityError:
        conn.rollback()

test_order_id_not_in_orders()
conn.close()